# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, following the Croissant standard and referencing all elements by their `@id` for full reproducibility and transparency.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install the necessary packages if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.date_published}\n")
print(f"Version: {metadata.version}\n")

## 2. Data Overview
Review available record sets, their field `@id`s, and understand the structure of the dataset.
**Note:** All Croissant elements are referenced using their `@id`.


In [ ]:
# List available record sets
print("Available record sets (by @id):")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- {record_set.id} (name: {record_set.name})")
    record_sets.append(record_set.id)
    # List all fields in this record set
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.id} (name: {field.name}, dataType: {field.data_type})")
    print()
# For most FAIR² datasets the table is typically the main record set, use the first as default
if len(record_sets) > 0:
    default_record_set_id = record_sets[0]
else:
    default_record_set_id = None

### Preview example records (using a record set's `@id`)


In [ ]:
# Preview records in the main record set (by @id)
# You may change the record_set_id variable to explore other record sets.
record_set_id = default_record_set_id

if record_set_id is not None:
    print(f"Showing 2 example records from record set {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i > 0:
            break
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load all records from the main record set(s) into DataFrames for analysis, referencing record sets by their `@id`.


In [ ]:
# Load each record set into a DataFrame for analysis
dataframes = {}
for rs_id in record_sets:
    print(f"Loading records for record set {rs_id}...")
    rows = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(rows)
    dataframes[rs_id] = df

# Show columns/fields for the default/main record set
if default_record_set_id is not None:
    print(f"\nColumns for {default_record_set_id}:")
    print(dataframes[default_record_set_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[default_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering numeric fields, normalization, categorization, and grouping.
We reference fields using their Croissant `@id` and suggest typical EDA steps for quantitative omics data.

In [ ]:
# -- Choose a numeric field for analysis (by @id) --
# Review previous printout for the most relevant field names and IDs in your dataset.
numeric_field_id = None
group_field_id = None

# Attempt to guess a reasonable numeric field for demonstration
df = dataframes[default_record_set_id]
for c in df.columns:
    if ("abundance" in c.lower()) or ("intensity" in c.lower()):
        numeric_field_id = c
        break

# Fallback: choose first float or int column
if numeric_field_id is None:
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break

# Try to find a categorical/group field (e.g. 'description' or 'accession')
for c in df.columns:
    if ("description" in c.lower()) or ("gene" in c.lower()) or ("accession" in c.lower()):
        group_field_id = c
        break

print(f"Using numeric field: {numeric_field_id}")
if group_field_id: print(f"Grouping field suggested: {group_field_id}")

# EDA: filter, normalize, group
if numeric_field_id is not None:
    # Filter: remove rows with missing or outlier values
    df_num = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
    df_num[numeric_field_id] = df_num[numeric_field_id].astype(float)
    threshold = df_num[numeric_field_id].quantile(0.75)  # Example: top quartile
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (Z-score, filtered records):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group (if applicable)
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped aggregate (mean) by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for automated EDA. Please review dataset columns.")

## 5. Visualization
Visualize the distribution of a selected numeric field and relationships with categorical or group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
if numeric_field_id is not None and numeric_field_id in df:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_num[numeric_field_id], bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If group_field_id is set, plot top 10 groups by mean value
if group_field_id and group_field_id in df_num:
    group_means = df_num.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    top10 = group_means.head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=top10.index, y=top10.values)
    plt.title(f"Top 10 {group_field_id}s by mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates end-to-end access, referencing by Croissant `@id`, and exploratory analysis of the FAIR² Human Mast Cell Mass Spec dataset using the `mlcroissant` Python library. 

- The dataset exposes its main tabular data as record sets, whose fields map to biological and technical features (e.g. protein abundance, accession, gene, description).
- Numeric exploration and basic visualization allow deeper insights and hypotheses for downstream analysis.

For domain-specific research, continue by integrating with other bioinformatics tools or performing downstream machine learning tasks with the processed data frames presented above.